In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arcpy
from arcpy.ia import *
from arcpy import AIO
import itertools
import os
from arcpy.ia import Clip as iaClip

In [2]:
# bounding box dla zbiornika Asprokremmos
bbox = [32.53, 34.724685, 32.578648, 34.762221] 

# zakres dat 
date_range = "2025-06-06T00:00:00Z/2026-06-01T23:59:59Z"

# zakresy dla zapytań STAC
date_range_old = "2025-06-01T00:00:00Z/2025-06-09T23:59:59Z"
date_range_new = "2026-06-01T00:00:00Z/2026-06-08T23:59:59Z"

# maksymalne pokrycie chmur
max_cloud_cover = 5

In [3]:
# starsza scena (np. lato 2025)
query_old = {
    "collections": ["sentinel-2-l2a"],
    "bbox": bbox,
    "query": {"eo:cloud_cover": {"lt": max_cloud_cover}},
    "datetime": date_range_old,
    "limit": 1  # 1 najlepsze zdjęcie
}

# nowa scena (np. lato 2026)
query_new = {
    "collections": ["sentinel-2-l2a"],
    "bbox": bbox,
    "query": {"eo:cloud_cover": {"lt": max_cloud_cover}},
    "datetime": date_range_new,
    "limit": 1  # 1 najlepsze zdjęcie
}

# pobieranie kolekcji z pełnym zestawem pasm
rc_old = arcpy.ia.RasterCollection.fromSTACAPI("https://earth-search.aws.element84.com/v1", 
                                               query=query_old, 
                                               attribute_dict={"Name":"id", "StdTime":"datetime"})
rc_new = arcpy.ia.RasterCollection.fromSTACAPI("https://earth-search.aws.element84.com/v1", 
                                               query=query_new, 
                                               attribute_dict={"Name":"id", "StdTime":"datetime"})


In [4]:
raster_old = rc_old[0]["Raster"]
raster_new = rc_new[0]["Raster"]

Przycięcie bandów po kolei

In [5]:
sr_wgs84 = arcpy.SpatialReference(4326)
sr_raster = raster_old.spatialReference

pt_min = arcpy.Point(bbox[0], bbox[1])
pt_max = arcpy.Point(bbox[2], bbox[3])

pt_min_proj = arcpy.PointGeometry(pt_min, sr_wgs84).projectAs(sr_raster)
pt_max_proj = arcpy.PointGeometry(pt_max, sr_wgs84).projectAs(sr_raster)

extent_proj = arcpy.Extent(
    pt_min_proj.firstPoint.X,
    pt_min_proj.firstPoint.Y,
    pt_max_proj.firstPoint.X,
    pt_max_proj.firstPoint.Y,
    spatial_reference=sr_raster
)

Co ArcPy widzi jako nazwy pasm

In [6]:
clipped_old = iaClip(raster_old, extent_proj)
band_names = clipped_old.bandNames
print("Wykryte pasma w przyciętym rastrze:", band_names)

Wykryte pasma w przyciętym rastrze: ['Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5', 'Band_6', 'Band_7', 'Band_8', 'Band_9', 'Band_10', 'Band_11', 'Band_12', 'Band_13', 'Band_14', 'Band_15', 'Band_16', 'Band_17', 'Band_18', 'Band_19', 'Band_20', 'Band_21', 'Band_22', 'Band_23', 'Band_24', 'Band_25', 'Band_26', 'Band_27', 'Band_28', 'Band_29', 'Band_30']


Eksport do PNG

In [8]:
out_dir_png = r"c:/temp/bandy_png_clip"
os.makedirs(out_dir_png, exist_ok=True)

# Pobieramy macierz tylko dla wyciętego obszaru
arr_full = arcpy.RasterToNumPyArray(clipped_old, nodata_to_value=0)

for i, b_name in enumerate(band_names):
    data = arr_full[i]
    
    # Skalowanie obrazu (2-98 percentyl dla lepszego kontrastu w odcieniach szarości)
    if np.any(data > 0):
        p2, p98 = np.percentile(data[data > 0], (2, 98))
    else:
        p2, p98 = 0, 1
        
    if p98 > p2:
        img_scaled = np.clip((data - p2) / (p98 - p2 + 1e-6), 0, 1)
    else:
        img_scaled = np.zeros_like(data)
        
    png_path = os.path.join(out_dir_png, f"band_{i+1:02d}_{b_name}.png")
    plt.imsave(png_path, img_scaled, cmap="gray")

Kompozycje RGB

In [10]:
# selected = [3, 5, 7, 11, 12]  
selected = [3, 5, 7, 8, 15]  

# Zabezpieczenie przed podaniem indeksu wykraczającego poza liczbę pasm
selected = [b for b in selected if b <= len(band_names)]
idx_python = [b - 1 for b in selected]

# Permutacje kanałów (R, G, B)
perms = list(itertools.permutations(range(len(idx_python)), 3))

out_dir_rgb = r"c:/temp/rgb_kompozycje_clip"
os.makedirs(out_dir_rgb, exist_ok=True)

for perm in perms:
    r_idx = idx_python[perm[0]]
    g_idx = idx_python[perm[1]]
    b_idx = idx_python[perm[2]]
    
    # Wyciągamy wybrane 3 kanały
    rgb = arr_full[[r_idx, g_idx, b_idx], :, :].astype(np.float32)

    # Normalizacja każdego kanału z osobna dla prawidłowych kolorów
    for c in range(3):
        p2, p98 = np.percentile(rgb[c], (2, 98))
        if p98 > p2:
            rgb[c] = np.clip((rgb[c] - p2) / (p98 - p2 + 1e-6), 0, 1)
        else:
            rgb[c] = 0

    # Formatowanie osi dla matplotlib (Y, X, Channels)
    rgb_img = np.transpose(rgb, (1, 2, 0))

    # Nazwa pliku odzwierciedla realne numery pasm (1-based) ze zmiennej 'selected'
    filename = f"RGB_R{selected[perm[0]]}_G{selected[perm[1]]}_B{selected[perm[2]]}.png"
    filepath = os.path.join(out_dir_rgb, filename)

    plt.imsave(filepath, rgb_img)

NDWI

In [ ]:
# Twoje bandy 1-based
selected = [3, 5, 7, 8, 15]

# Zamiana na indeksy Pythona
idx_python = [b - 1 for b in selected]

# Wszystkie pary (green, nir)
pairs = list(itertools.permutations(range(len(idx_python)), 2))

# Folder wyjściowy
out_dir_ndwi = r"c:/temp/ndwi_test"
os.makedirs(out_dir_ndwi, exist_ok=True)

for g_i, n_i in pairs:
    # indeksy Pythona
    g_idx = idx_python[g_i]
    n_idx = idx_python[n_i]

    # Numery bandów 1-based (do nazw plików)
    g_band = selected[g_i]
    n_band = selected[n_i]

    # Pobranie danych
    green = arr_full[g_idx].astype(np.float32)
    nir   = arr_full[n_idx].astype(np.float32)

    # NDWI
    ndwi = (green - nir) / (green + nir + 1e-6)

    # Normalizacja do 0–1
    ndwi_norm = (ndwi - ndwi.min()) / (ndwi.max() - ndwi.min() + 1e-6)

    # Nazwa pliku
    filename = f"NDWI_G{g_band}_N{n_band}.png"
    filepath = os.path.join(out_dir_ndwi, filename)

    plt.imsave(filepath, ndwi_norm, cmap="grey")


NDVI

In [ ]:
# poprawne pary (Red, NIR) w numeracji 1-based
valid_pairs = [
    (3, 7),
    (3, 15),
    (5, 7),
    (5, 15),
    (7, 15),
    (8, 7),
    (8, 15),
]

out_dir_ndvi = r"c:/temp/ndvi_test"
os.makedirs(out_dir_ndvi, exist_ok=True)

for r_band, n_band in valid_pairs:
    r_idx = r_band - 1
    n_idx = n_band - 1

    red = arr_full[r_idx].astype(np.float32)
    nir = arr_full[n_idx].astype(np.float32)

    ndvi = (nir - red) / (nir + red + 1e-6)

    ndvi_norm = (ndvi - ndvi.min()) / (ndvi.max() - ndvi.min() + 1e-6)

    filename = f"NDVI_R{r_band}_N{n_band}.png"
    filepath = os.path.join(out_dir_ndvi, filename)

    plt.imsave(filepath, ndvi_norm, cmap="grey")


DOTĄD WAŻNE ---------------------------------------------------------

In [ ]:
def to_rgb(arr):
    """
    Konwersja z automatyczną normalizacją (min-max opartą na percentylach dla każdego kanału).
    Wejście: arr o kształcie (4, Y, X), gdzie indeksy to: 0->Red, 1->Green, 2->Blue, 3->NIR
    """
    # Wybieramy pierwsze trzy kanały: Red (0), Green (1), Blue (2)
    rgb = arr[[0, 1, 2], :, :].astype(np.float32)
    
    # Normalizacja każdego kanału osobno do zakresu [0, 1]
    for i in range(3):
        ch = rgb[i, :, :]
        # Używamy percentyli, aby odrzucić szum/chmury (2% i 98%)
        p2, p98 = np.percentile(ch, (2, 98))
        if p98 > p2:
            rgb[i, :, :] = np.clip((ch - p2) / (p98 - p2), 0, 1)
        else:
            rgb[i, :, :] = 0  # zabezpieczenie przed dzieleniem przez zero przy braku zróżnicowania pikseli
        
    # Transpozycja z (Channels, Rows, Columns) do (Rows, Columns, Channels) dla matplotlib
    return np.transpose(rgb, (1, 2, 0))

date_old = str(rc_old[0]["StdTime"])[:10]
date_new = str(rc_new[0]["StdTime"])[:10]

# Generowanie obrazów RGB tylko dla zoptymalizowanych, przyciętych danych
rgb_old = to_rgb(arr_old)
rgb_new = to_rgb(arr_new)

# Tworzymy wykres: 1 wiersz, 2 kolumny (porównanie lat dla przyciętego bboxu)
fig, axes = plt.subplots(1, 2, figsize=(14, 7), facecolor="#0f1117")
fig.suptitle(
    "Zbiornik Asprokremmos — Kompozycja RGB\nPorównanie obszaru przyciętego do BBOX",
    color="white", fontsize=14, fontweight="bold", y=0.98
)

panels = [
    (axes[0], rgb_old, f"Przycięty — {date_old}"),
    (axes[1], rgb_new, f"Przycięty — {date_new}"),
]

for ax, img, title in panels:
    ax.imshow(img, interpolation="bilinear")
    ax.set_title(title, color="white", fontsize=12, pad=10)
    ax.axis("off")
    ax.set_facecolor("#0f1117")

plt.tight_layout()
plt.show()

# Kontrola rozmiarów (teraz operujemy na małych, szybko pobierających się macierzach)
print(f"Przycięty 2025: {arr_old.shape}  →  Obraz wynikowy RGB: {rgb_old.shape[:2]} px")
print(f"Przycięty 2026: {arr_new.shape}  →  Obraz wynikowy RGB: {rgb_new.shape[:2]} px")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Red (B4)", "Green (B3)", "Blue (B2)", "NIR (B8)"]

for i in range(4):
    # Prosty rzut w skali szarości. 
    # vmax=3000 zapobiega sytuacji, w której obraz jest całkowicie czarny przez odblaski/chmury.
    axes[i].imshow(arr_new[i], cmap='gray', vmin=0, vmax=3000)
    axes[i].set_title(titles[i])
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Generujemy 24 kombinacje (wybieramy 3 pasma z 4 dostępnych)
perms = list(itertools.permutations([0, 1, 2, 3], 3))

fig, axes = plt.subplots(4, 6, figsize=(22, 14), facecolor="#0f1117")
fig.suptitle("Wszystkie 24 możliwe kompozycje RGB z 4 pobranych pasm", 
             color="white", fontsize=18, fontweight="bold", y=0.98)

for (perm, ax) in zip(perms, axes.flatten()):
    # Wyciągamy kanały w aktualnie testowanej kolejności
    test_rgb = arr_new[list(perm), :, :].astype(np.float32)
    
    # Szybka normalizacja
    for c in range(3):
        p2, p98 = np.percentile(test_rgb[c], (2, 98))
        if p98 > p2:
            test_rgb[c] = np.clip((test_rgb[c] - p2) / (p98 - p2), 0, 1)
        else:
            test_rgb[c] = 0
            
    # Zamiana osi dla Matplotlib (Y, X, Channels)
    test_rgb = np.transpose(test_rgb, (1, 2, 0))
    
    ax.imshow(test_rgb)
    # Tytuł pokaże nam dokładnie, który indeks użyto do którego koloru
    ax.set_title(f"R:{perm[0]} G:{perm[1]} B:{perm[2]}", color="white", fontsize=11, pad=6)
    ax.axis("off")
    ax.set_facecolor("#0f1117")

plt.tight_layout()
plt.show()

In [ ]:
def adjust_rgb(arr):
    ref = arr * 0.0000275 - 0.2
    ref = np.clip(ref, 0, None)
    vmax = 0.3  
    return np.clip(ref / vmax, 0, 1)

s_r_old = adjust_rgb(arr_old[[0],:,:])
s_g_old = adjust_rgb(arr_old[[1],:,:])
s_b_old = adjust_rgb(arr_old[[2],:,:])

s_r_new = adjust_rgb(arr_new[[0],:,:])
s_g_new = adjust_rgb(arr_new[[1],:,:])
s_b_new = adjust_rgb(arr_new[[2],:,:])

# rgb_old = np.stack([s_r_old, s_g_old, s_b_old], axis=-1)
# rgb_new = np.stack([s_r_new, s_g_new, s_b_new], axis=-1)

# rgb_old = rgb_old.squeeze()
# rgb_new = rgb_new.squeeze()

NDSI

In [ ]:
# konwersja na float, aby zapobiec błędom dzielenia i przepełnienia typów
g_old_f = arr_old[[1],:,:].astype(float)
g_new_f = arr_new[[1],:,:].astype(float)

nir_old_f = arr_old[[3],:,:].astype(float)
nir_new_f = arr_new[[3],:,:].astype(float)

denom_old = g_old_f + nir_old_f
denom_new = g_new_f + nir_new_f

# bezpieczne dzielenie (tam gdzie mianownik to 0, wpisujemy 0)
ndwi_old = np.where(denom_old == 0, 0, (g_old_f - nir_old_f) / denom_old)
ndwi_new = np.where(denom_new == 0, 0, (g_new_f - nir_new_f) / denom_new)

In [ ]:
# maska prawidłowych danych (wykluczamy czarne rogi rombu)
valid_data_old = denom_old > 0
valid_data_new = denom_new > 0

# generowanie binarnej maski (True dla śniegu/lodu, False dla reszty)
snow_mask_old = ndwi_old > 0.0
snow_mask_new = ndwi_new > 0.0

snow_mask_old = np.squeeze(snow_mask_old)
snow_mask_new = np.squeeze(snow_mask_new)

snow_pixels_old = np.sum(snow_mask_old & valid_data_old)
total_pixels_old = np.sum(valid_data_old)
pct_snow_old = (snow_pixels_old / total_pixels_old) * 100 if total_pixels_old > 0 else 0

snow_pixels_new = np.sum(snow_mask_new & valid_data_new)
total_pixels_new = np.sum(valid_data_new)
pct_snow_new = (snow_pixels_new / total_pixels_new) * 100 if total_pixels_new > 0 else 0

In [ ]:
date_old = str(rc_old[0]["StdTime"])[:10]
date_new = str(rc_new[0]["StdTime"])[:10]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# pierwszy wiersz - RGB
axes[0, 0].imshow(rgb_old)
axes[0, 0].set_title(f"RGB - {date_old}", pad=10)
axes[0, 0].axis('off')

axes[0, 1].imshow(rgb_new)
axes[0, 1].set_title(f"RGB - {date_new}", pad=10)
axes[0, 1].axis('off')

axes[1, 0].imshow(snow_mask_old, cmap='Blues')
axes[1, 0].set_title(f"NDWI > 0.0 ({pct_snow_old:.2f}%) - {date_old}", pad=10)
axes[1, 0].axis('off')

axes[1, 1].imshow(snow_mask_new, cmap='Blues')
axes[1, 1].set_title(f"NDWI > 0.0 ({pct_snow_new:.2f}%) - {date_new}", pad=10)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
red_old   = arr_old[0, :, :].astype(float) / 10000.0
green_old = arr_old[1, :, :].astype(float) / 10000.0
blue_old  = arr_old[2, :, :].astype(float) / 10000.0
nir_old   = arr_old[3, :, :].astype(float) / 10000.0

red_new   = arr_new[0, :, :].astype(float) / 10000.0
green_new = arr_new[1, :, :].astype(float) / 10000.0
blue_new  = arr_new[2, :, :].astype(float) / 10000.0
nir_new   = arr_new[3, :, :].astype(float) / 10000.0

# %%
# 1. PRZYGOTOWANIE RGB
# Łączymy pasma R, G, B w jeden obraz i przycinamy do zakresu [0, 1]
# Opcjonalnie: mnożymy przez powiększenie (np. 2.5), aby rozjaśnić obraz
# vmax = 0.3  # maksymalny oczekiwany refleks dla wizualizacji (zapobiega prześwietleniu)
# rgb_old = np.clip(np.stack([red_old, green_old, blue_old], axis=-1) / vmax, 0, 1)
# rgb_new = np.clip(np.stack([red_new, green_new, blue_new], axis=-1) / vmax, 0, 1)


# %%
# 2. OBLICZENIE WSKAŹNIKA NDWI (dla wody: (Green - NIR) / (Green + NIR))
denom_old = green_old + nir_old
denom_new = green_new + nir_new

ndwi_old = np.where(denom_old == 0, 0, (green_old - nir_old) / denom_old)
ndwi_new = np.where(denom_new == 0, 0, (green_new - nir_new) / denom_new)


# %%
# 3. MASKOWANIE WODY (Zbiornik Asprokremmos)
# Dla czystej wody NDWI jest zazwyczaj dodatnie (> 0 lub > 0.1)
# Jeśli celowo szukasz śniegu za pomocą NDSI, formuła to (Green - SWIR) / (Green + SWIR), 
# ale nie pobrałeś pasma SWIR (B11) w selected_bands. Zakładam, że badasz wodę (NDWI).

water_threshold = 0.0  # Próg dla wody, możesz dostosować (np. 0.0 lub 0.2)

valid_data_old = denom_old > 0
valid_data_new = denom_new > 0

water_mask_old = ndwi_old > water_threshold
water_mask_new = ndwi_new > water_threshold

# Obliczenie procentu powierzchni wody
water_pixels_old = np.sum(water_mask_old & valid_data_old)
total_pixels_old = np.sum(valid_data_old)
pct_water_old = (water_pixels_old / total_pixels_old) * 100 if total_pixels_old > 0 else 0

water_pixels_new = np.sum(water_mask_new & valid_data_new)
total_pixels_new = np.sum(valid_data_new)
pct_water_new = (water_pixels_new / total_pixels_new) * 100 if total_pixels_new > 0 else 0


# %%
# 4. RYSOWANIE WYKRESÓW
date_old = str(rc_old[0]["StdTime"])[:10]
date_new = str(rc_new[0]["StdTime"])[:10]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Pierwszy wiersz - RGB
axes[0, 0].imshow(rgb_old)
axes[0, 0].set_title(f"RGB - {date_old}", pad=10)
axes[0, 0].axis('off')

axes[0, 1].imshow(rgb_new)
axes[0, 1].set_title(f"RGB - {date_new}", pad=10)
axes[0, 1].axis('off')

# Drugi wiersz - NDWI (Maska wody)
axes[1, 0].imshow(water_mask_old, cmap='Blues')
axes[1, 0].set_title(f"NDWI > {water_threshold} ({pct_water_old:.2f}%) - {date_old}", pad=10)
axes[1, 0].axis('off')

axes[1, 1].imshow(water_mask_new, cmap='Blues')
axes[1, 1].set_title(f"NDWI > {water_threshold} ({pct_water_new:.2f}%) - {date_new}", pad=10)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()